# NB_hospital_daily_summary
Builds **`<catalog>.gold.hospital_daily_summary`** from the silver layer, as defined in the `sttm_silver_to_gold.xlsx` STTM (sheet `hospital_daily_summary`).

* **Grain:** one row per hospital per calendar day
* **Driving table:** `silver.lab_results` grouped by `hospital_id`, `test_date`
* **Joins (LEFT):** `silver.hospitals` on `hospital_id`; `silver.calendar` on date key (optional, for `day_name` / `is_weekend`)
* **Window function:** `LAG(total_tests)` partitioned by `hospital_id`, ordered by `summary_date` (day-over-day trend)
* **Filters:** none
* **Load type:** FULL, `CREATE OR REPLACE TABLE ... AS SELECT`

Note: the STTM names the calendar key `date_key`; the actual column in `silver.calendar` is `datekey` (INT, yyyyMMdd), so the join uses `datekey = CAST(date_format(test_date, 'yyyyMMdd') AS INT)`.

In [0]:
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"

print("=" * 80)
print("STEP 1: Input parameters")
print("=" * 80)
print(f"  catalog       : '{catalog}'")
print(f"  Source schema : {source_schema}")
print(f"  Target table  : {target_table}")

In [0]:
hospital_daily_summary_sql = f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH daily AS (
  -- Lab results aggregated to hospital x day grain
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM {source_schema}.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  -- Day-over-day trend: previous day's volume for the same hospital
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  CAST(d.hospital_id AS INT)                                                AS hospital_id,
  h.hospital_name,
  d.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(d.total_tests AS INT)                                                AS total_tests,
  CAST(d.unique_patients_tested AS INT)                                     AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)                                        AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)                                        AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)                                        AS critical_test_count,
  CAST(d.abnormal_test_count * 100.0 / d.total_tests AS DECIMAL(5,2))      AS abnormal_rate_pct,
  CAST(d.critical_test_count * 100.0 / d.total_tests AS DECIMAL(5,2))      AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)                                       AS prev_day_total_tests,
  CASE
    WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
    ELSE CAST((d.total_tests - d.prev_day_total_tests) * 100.0 / d.prev_day_total_tests AS DECIMAL(6,2))
  END                                                                       AS test_volume_change_pct
FROM with_lag d
LEFT JOIN {source_schema}.hospitals h
  ON d.hospital_id = h.hospital_id
LEFT JOIN {source_schema}.calendar c
  ON c.datekey = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT)
"""

print("=" * 80)
print("STEP 2: Building gold table (FULL load)")
print("=" * 80)
print("Executing query:")
print(hospital_daily_summary_sql)

spark.sql(hospital_daily_summary_sql)
print(f"FULL load completed -> {target_table}")

In [0]:
from pyspark.sql import functions as F

print("=" * 80)
print("STEP 3: Validation")
print("=" * 80)

source_tests = spark.table(f"{source_schema}.lab_results").count()
target_df = spark.table(target_table)
target_count = target_df.count()
distinct_keys = target_df.select("hospital_id", "summary_date").distinct().count()
target_tests = target_df.agg(F.sum("total_tests")).collect()[0][0] or 0

print(f"  Target rows                       : {target_count}")
print(f"  Distinct (hospital_id, date)      : {distinct_keys}")
print(f"  Source lab_results rows           : {source_tests}")
print(f"  SUM(total_tests) in target        : {target_tests}")

if distinct_keys != target_count:
    raise ValueError("Grain violation: (hospital_id, summary_date) is not unique in target")
if target_tests != source_tests:
    raise ValueError("Test count mismatch between silver.lab_results and target")
print("Validation passed.")

#display(target_df.orderBy("hospital_id", "summary_date").limit(10))

In [0]:
record_count = target_df.count()
print(f"Records in {target_table}: {record_count}")
dbutils.notebook.exit(str(record_count))